In [ ]:
# ===== CLEAN START SCRIPT =====
# RUN THIS AFTER RESTARTING KERNEL!

import os
import sys
from pathlib import Path

print("🔄 Fresh Start - Checking environment...")
print("=" * 60)

# 1. Clear any existing GitHub env vars (just in case)
for key in list(os.environ.keys()):
    if 'GITHUB' in key:
        print(f"   Removing: {key}")
        del os.environ[key]

# 2. Check current environment (should be empty)
print(f"\n📊 Environment after clearing:")
github_vars = {k: v for k, v in os.environ.items() if 'GITHUB' in k}
if github_vars:
    for k, v in github_vars.items():
        print(f"   {k}: {v[:20]}...")
else:
    print("   ✅ No GitHub environment variables found")

# 3. Now load from .env properly
from dotenv import load_dotenv

env_path = Path('.').resolve() / '.env'
print(f"\n🔍 Loading .env from: {env_path}")

if env_path.exists():
    # Load with override to ensure .env wins
    load_dotenv(dotenv_path=env_path, override=True)
    
    # Verify
    token = os.getenv('GITHUB_TOKEN')
    username = os.getenv('GITHUB_USERNAME')
    
    print(f"✅ .env loaded successfully!")
    print(f"   Token from .env: {token[:15]}..." if token else "   ❌ No token in .env")
    print(f"   Username from .env: {username}")
    
    if token and token.startswith('ghp_'):
        print("   ✅ Token looks valid (starts with 'ghp_')")
    elif token and 'your_actual' in token:
        print("   ❌ Token contains placeholder text!")
    else:
        print("   ⚠️  Token format unexpected")
else:
    print(f"❌ .env file not found!")

print("\n✅ Environment is now clean and loaded from .env!")

# ===== MAIN SCRIPT (run after the clean start) =====
import requests
import json
import subprocess

print("🚀 GitHub Repository Creator")
print("=" * 60)

# Get values from environment (now properly loaded from .env)
token = os.getenv('GITHUB_TOKEN')
username = os.getenv('GITHUB_USERNAME')
repo_name = os.getenv('REPO_NAME', 'Data_engineering_journey')
description = os.getenv('REPO_DESCRIPTION', 'Learning data engineering from scratch')
is_private = os.getenv('IS_PRIVATE', 'false').lower() == 'true'

print(f"📋 Configuration:")
print(f"   User: {username}")
print(f"   Token: {token[:10]}... (length: {len(token)})")
print(f"   Repo: {repo_name}")
print(f"   Description: {description}")
print(f"   Private: {is_private}")

# Test the token
print(f"\n🔐 Testing GitHub connection...")
headers = {
    "Authorization": f"token {token}",
    "Accept": "application/vnd.github.v3+json",
    "User-Agent": "Jupyter-Notebook-App"
}

response = requests.get("https://api.github.com/user", headers=headers)

if response.status_code == 200:
    user_data = response.json()
    print(f"✅ Connected as: {user_data.get('login')}")
    
    # Create repository
    print(f"\n📦 Creating repository: {repo_name}")
    
    create_data = {
        "name": repo_name,
        "description": description,
        "private": is_private,
        "auto_init": False
    }
    
    create_response = requests.post(
        "https://api.github.com/user/repos",
        headers=headers,
        json=create_data
    )
    
    if create_response.status_code == 201:
        repo_data = create_response.json()
        print(f"✅ Repository created!")
        print(f"🔗 URL: {repo_data.get('html_url')}")
        
        # Setup git
        print(f"\n💻 Setting up Git...")
        remote_url = f"git@github.com:{username}/{repo_name}.git"
        
        try:
            subprocess.run(["git", "remote", "add", "origin", remote_url], check=True)
            print(f"   ✅ Remote added: {remote_url}")
            
            subprocess.run(["git", "branch", "-M", "main"], check=True)
            print(f"   ✅ Branch renamed to 'main'")
            
            print(f"   🚀 Pushing code...")
            push_result = subprocess.run(
                ["git", "push", "-u", "origin", "main"],
                capture_output=True,
                text=True
            )
            
            if push_result.returncode == 0:
                print(f"   ✅ Code pushed to GitHub!")
            else:
                print(f"   ⚠️  Push output: {push_result.stderr}")
                
        except Exception as e:
            print(f"   ⚠️  Git error: {e}")
            print(f"\n💡 Manual commands:")
            print(f"   git remote add origin {remote_url}")
            print(f"   git branch -M main")
            print(f"   git push -u origin main")
            
    elif create_response.status_code == 422:
        error_data = create_response.json()
        if 'errors' in error_data:
            for error in error_data['errors']:
                if error.get('code') == 'already_exists':
                    print(f"✅ Repository already exists!")
                    print(f"🔗 https://github.com/{username}/{repo_name}")
                    print(f"\n💡 To push your code:")
                    print(f"   git remote add origin git@github.com:{username}/{repo_name}.git")
                    print(f"   git branch -M main")
                    print(f"   git push -u origin main")
                    break
            else:
                print(f"❌ Validation error: {json.dumps(error_data, indent=2)}")
        else:
            print(f"❌ Error: {create_response.text[:200]}")
            
    else:
        print(f"❌ Creation failed: {create_response.status_code}")
        print(f"   Error: {create_response.text[:200]}")
        
else:
    print(f"❌ Authentication failed: {response.status_code}")
    print(f"   Error: {response.text[:200]}")

# This is just to check if if version control will work !

🔄 Fresh Start - Checking environment...
   Removing: GITHUB_TOKEN
   Removing: GITHUB_USERNAME

📊 Environment after clearing:
   ✅ No GitHub environment variables found

🔍 Loading .env from: /home/odinsbeard/Data_engineering_Journey/.env
✅ .env loaded successfully!
   Token from .env: ghp_7r0Xy6ljuhs...
   Username from .env: Ea-mjolnir
   ✅ Token looks valid (starts with 'ghp_')

✅ Environment is now clean and loaded from .env!
🚀 GitHub Repository Creator
📋 Configuration:
   User: Ea-mjolnir
   Token: ghp_7r0Xy6... (length: 40)
   Repo: Data_engineering_journey
   Description: Learning data engineering from scratch
   Private: False

🔐 Testing GitHub connection...
✅ Connected as: Ea-mjolnir

📦 Creating repository: Data_engineering_journey
✅ Repository created!
🔗 URL: https://github.com/Ea-mjolnir/Data_engineering_journey

💻 Setting up Git...
   ✅ Remote added: git@github.com:Ea-mjolnir/Data_engineering_journey.git
   ✅ Branch renamed to 'main'
   🚀 Pushing code...
   ✅ Code pushed to 